In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scite.beam.continuous.continuous_beam_var_cs import ContinuousBeamVarCSModes


## Moment redistribution factor δ vs neutral-axis depth x/d

This notebook studies the **achievable redistribution factor**

$$\delta_{M,u} = \frac{|M_{R,\mathrm{hog}}|}{|M^\mathrm{el}_\mathrm{hog}(q_u)|}$$

where $M_{R,\mathrm{hog}}$ is the hogging moment capacity of the support section and
$M^\mathrm{el}_\mathrm{hog}(q_u) = q_u L^2/8$ is the elastic reference hogging moment
for the same load on an equal-span two-span beam.

$\delta_{M,u} < 1$ means the beam has redistributed load from the hogging zone to
the sagging spans before the hogging section reaches its capacity.

### EC2 §5.5 limit (German NA, B500B, C ≤ C50/60)

$$\delta \ge \underbrace{0.64 + 0.80\,\frac{x_u}{d}}_{\text{linear constraint}}
  \quad \text{and} \quad \delta \ge 0.80$$

This is a **lower bound on the permissible** δ.  
Sections with $(x_u/d, \delta)$ lying **below** this line cannot sustain the
proposed redistribution and must be redesigned.

The simulation computes $\delta_{M,u}$ from the nonlinear CBA sweep using the same
section parameters as notebook 03.


In [ ]:
# ── Geometry ──────────────────────────────────────────────────────────────────
b, h    = 300.0, 700.0
z_s_bot = 50.0
z_s_top = h - z_s_bot      # 650 mm from bottom
d_hog   = z_s_top           # effective depth at hogging section
L       = 6000.0

# ── Design material values ────────────────────────────────────────────────────
f_ck, f_yk        = 30.0, 500.0
gamma_c, gamma_s, alpha_cc = 1.5, 1.15, 0.85
f_cd = alpha_cc * f_ck / gamma_c   # = 17.0 MPa
f_yd = f_yk / gamma_s              # ≈ 434.78 MPa

# ── A_s range derived from target x/d values ─────────────────────────────────
# x/d = omega/0.8  with  omega = rho * f_yd / f_cd
# → A_s = x/d * 0.8 * (f_cd / f_yd) * b * d
xd_targets = np.linspace(0.05, 0.50, 7)
A_s_range  = xd_targets * 0.8 * (f_cd / f_yd) * b * d_hog

print(f'Design:  f_cd = {f_cd:.2f} MPa,  f_yd = {f_yd:.2f} MPa,  d_hog = {d_hog:.0f} mm')
print()
print(f'{"x/d target":>12}  {"A_s [mm²]":>12}')
for xd, As in zip(xd_targets, A_s_range):
    print(f'{xd:12.3f}  {As:12.0f}')

# ── EC2 §5.5 reference curve ──────────────────────────────────────────────────
xd_ec2_line    = np.linspace(0.0, 0.50, 200)
delta_ec2_min  = np.maximum(0.64 + 0.80 * xd_ec2_line, 0.80)

COLOR = {'design': '#d62728', 'mean': '#2ca02c'}
STYLE = {'design': '--', 'mean': '-'}


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def hog_failure_mode(cba) -> str:
    """Failure mode of the hogging section at its M-κ peak.

    Mirrors the logic of MKappaAnalysis._add_failure_marker_neg.

    Returns
    -------
    'concrete'      η_c > η_f  — concrete crushing at bottom fibre
    'reinforcement' η_f > η_c  — steel tension limit at top fibre
    'balanced'      |η_c - η_f| < 0.05
    """
    mk = cba.span_left.mk
    if mk.kappa_neg_values is None or len(mk.kappa_neg_values) == 0:
        return 'unknown'

    idx_pk     = int(np.argmin(mk.M_neg_values))
    kappa_pk   = float(mk.kappa_neg_values[idx_pk])
    eps_bot_pk = float(mk.eps_bot_neg_values[idx_pk])
    eps_top_pk = eps_bot_pk - kappa_pk * float(mk.cs.h_total)

    eps_cu = abs(float(mk.cs.concrete.eps_cu2_computed))
    top_layer = (max(mk.cs.reinforcement.layers, key=lambda l: l.z)
                 if mk.cs.reinforcement.layers else None)
    eps_ud_top = (getattr(top_layer.material, 'eps_ud', None)
                  or getattr(top_layer.material, 'eps_u', 0.025)
                  if top_layer else 0.025)

    eta_c = abs(eps_bot_pk) / eps_cu     if eps_cu    > 0 else float('nan')
    eta_f = eps_top_pk      / eps_ud_top if eps_ud_top > 0 else float('nan')

    if abs(eta_c - eta_f) < 0.05:
        return 'balanced'
    return 'concrete' if eta_c > eta_f else 'reinforcement'


# Marker shape and display label per failure mode
FM_MARKER = {'concrete': 's', 'reinforcement': '^', 'balanced': 'D'}
FM_LABEL  = {
    'concrete':      'Concrete crushing',
    'reinforcement': 'Steel tension limit',
    'balanced':      'Balanced failure',
}


def make_legend_handles():
    """Proxy handles for the combined legend (line style + failure mode marker)."""
    line_h = [
        Line2D([0], [0], color=COLOR['design'], ls='--', lw=2.2, label='Design'),
        Line2D([0], [0], color=COLOR['mean'],   ls='-',  lw=2.2, label='Mean'),
    ]
    fm_h = [
        Line2D([0], [0], marker=FM_MARKER[fm], color='0.45', ls='none',
               markersize=8, markeredgecolor='k', markeredgewidth=0.8,
               label=FM_LABEL[fm])
        for fm in ('concrete', 'reinforcement')
    ]
    ec2_h = [
        Line2D([0], [0], color='#1f77b4', lw=1.8,
               label=r'EC2 §5.5 $\delta_\mathrm{min}$'),
        Patch(facecolor='#1f77b4', alpha=0.15,
              label='EC2 §5.5 not permitted'),
    ]
    return ec2_h + line_h + fm_h


## Study 1 — Symmetric reinforcement ($A_{s,bot} = A_{s,top}$)

Same parametric range as Study 1 in notebook 03.  Both sagging and hogging
reinforcement areas are kept equal and swept together.


In [ ]:
results1 = {'design': [], 'mean': []}

for A_s in A_s_range:
    triple = ContinuousBeamVarCSModes.for_rc(
        b=b, h=h,
        A_s_bot=A_s, z_s_bot=z_s_bot,
        A_s_top=A_s, z_s_top=z_s_top,
        f_ck=f_ck, f_yk=f_yk,
        L_a=L, L_b=L,
        n_kappa=100,
    )
    print(f'A_s = {A_s:.0f} mm²')
    for mode_name, cba in [('design', triple.design), ('mean', triple.mean)]:
        xd           = cba.xd_hog_u
        delta        = cba.delta_M_u
        failure_mode = cba.failure_mode_u
        mk_fail      = hog_failure_mode(cba)
        q_u          = cba.q_hog_u
        M_nl         = abs(cba.M_nl_hog_u)   # kN·m
        M_el         = abs(cba.M_el_hog_u)   # kN·m
        results1[mode_name].append(dict(
            A_s=A_s, xd=xd, delta=delta,
            failure_mode=failure_mode, mk_fail=mk_fail,
            q_u=q_u, M_nl_hog=M_nl, M_el_hog=M_el,
        ))
        print(f'  {mode_name:12s}  x/d={xd:.4f}  δ={delta:.4f}  '
              f'M_nl={M_nl:.1f} kN·m  M_el={M_el:.1f} kN·m  '
              f'mode={failure_mode}  mk_fail={mk_fail}  q={q_u:.2f} kN/m')
    print()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.fill_between(xd_ec2_line, 0.45, delta_ec2_min,
                alpha=0.12, color='#1f77b4')
ax.plot(xd_ec2_line, delta_ec2_min, color='#1f77b4', lw=1.8)

for mode in ('design', 'mean'):
    r = results1[mode]
    xd_arr = np.array([pt['xd']      for pt in r])
    d_arr  = np.array([pt['delta']   for pt in r])
    fm_arr = np.array([pt['mk_fail'] for pt in r])

    ax.plot(xd_arr, d_arr, color=COLOR[mode], ls=STYLE[mode], lw=2.2, zorder=2)

    for fm in ('concrete', 'reinforcement', 'balanced'):
        mask = fm_arr == fm
        if mask.any():
            ax.scatter(xd_arr[mask], d_arr[mask],
                       marker=FM_MARKER.get(fm, 'o'), s=55,
                       facecolors=COLOR[mode], edgecolors='k', linewidths=0.8,
                       zorder=4)

ax.axhline(1.0, color='0.4', lw=1.0, ls=':', label=r'$\delta = 1$: elastic reference')
ax.axhline(0.80, color='#1f77b4', lw=0.8, ls=':', alpha=0.5)
ax.text(0.01, 0.805, r'$\delta = 0.80$ floor', fontsize=8, color='#1f77b4', va='bottom')

ax.set_xlabel(r'$x_u/d$  (hogging section)')
ax.set_ylabel(r'$\delta_{M,u} = |M_{R,\mathrm{hog}}| \;/\; |M^\mathrm{el}_\mathrm{hog}(q_u)|$')
ax.set_xlim(-0.01, 0.53)
ax.set_ylim(0.45, 1.12)
ax.legend(handles=make_legend_handles(), fontsize=9, loc='lower right')
ax.set_title(
    r'Study 1: Symmetric $A_s$ — achievable $\delta_{M,u}$ vs EC2 §5.5 limit'
    f'\n$b={b:.0f}$, $h={h:.0f}$ mm, C{f_ck:.0f}/37, B500B, $L={L/1000:.0f}$ m'
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Observations — Study 1

- **Both curves lie above the EC2 §5.5 lower bound** for $x_u/d \gtrsim 0.08$, meaning the
  simulation confirms the section can achieve the redistribution EC2 requires.
- At very small $x_u/d$ (lightly reinforced, design mode), $\delta_{M,u} \approx 0.75 < 0.80$:
  the section naturally redistributes *more* than the EC2 floor permits in the implicit
  §5.5 approach.  EC2 §5.5 imposes $\delta \ge 0.80$ as a conservative design limit, not
  a physical bound.
- **Design vs mean gap**: design-level material factors (γ_c = 1.5, γ_s = 1.15) reduce
  $M_{R,\mathrm{hog}}$ relative to mean; the same x/d therefore gives a lower δ.
- The curves converge toward $\delta \approx 1$ at large $x_u/d$ because a heavily
  reinforced hogging section is stiff enough to track the elastic moment distribution.


## Study 2 — Fixed sagging reinforcement, varying hogging reinforcement

$A_{s,bot}$ is held fixed at the mid-range value from Study 1 while
$A_{s,top}$ (and hence $x_u/d$ at the hogging section) is varied over the
same target range.  This isolates the hogging section's contribution to
redistribution from any change in sagging zone capacity.


In [ ]:
A_s_bot_fixed = float(A_s_range[len(A_s_range) // 2])   # mid-range sagging area
A_s_top_range = xd_targets * 0.8 * (f_cd / f_yd) * b * d_hog

print(f'Fixed sagging A_s_bot = {A_s_bot_fixed:.0f} mm²  '
      f'(x/d_sag ≈ {A_s_bot_fixed / (b * d_hog) * f_yd / f_cd / 0.8:.3f})')
print()

results2 = {'design': [], 'mean': []}

for A_s_top in A_s_top_range:
    triple = ContinuousBeamVarCSModes.for_rc(
        b=b, h=h,
        A_s_bot=A_s_bot_fixed, z_s_bot=z_s_bot,
        A_s_top=A_s_top,       z_s_top=z_s_top,
        f_ck=f_ck, f_yk=f_yk,
        L_a=L, L_b=L,
        n_kappa=100,
    )
    print(f'A_s_top = {A_s_top:.0f} mm²')
    for mode_name, cba in [('design', triple.design), ('mean', triple.mean)]:
        xd           = cba.xd_hog_u
        delta        = cba.delta_M_u
        failure_mode = cba.failure_mode_u
        mk_fail      = hog_failure_mode(cba)
        q_u          = cba.q_hog_u
        M_nl         = abs(cba.M_nl_hog_u)
        M_el         = abs(cba.M_el_hog_u)
        results2[mode_name].append(dict(
            A_s_top=A_s_top, xd=xd, delta=delta,
            failure_mode=failure_mode, mk_fail=mk_fail,
            q_u=q_u, M_nl_hog=M_nl, M_el_hog=M_el,
        ))
        print(f'  {mode_name:12s}  x/d={xd:.4f}  δ={delta:.4f}  '
              f'M_nl={M_nl:.1f} kN·m  M_el={M_el:.1f} kN·m  '
              f'mode={failure_mode}  mk_fail={mk_fail}  q={q_u:.2f} kN/m')
    print()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

ax.fill_between(xd_ec2_line, 0.45, delta_ec2_min,
                alpha=0.12, color='#1f77b4')
ax.plot(xd_ec2_line, delta_ec2_min, color='#1f77b4', lw=1.8)

for mode in ('design', 'mean'):
    r = results2[mode]
    xd_arr = np.array([pt['xd']      for pt in r])
    d_arr  = np.array([pt['delta']   for pt in r])
    fm_arr = np.array([pt['mk_fail'] for pt in r])

    ax.plot(xd_arr, d_arr, color=COLOR[mode], ls=STYLE[mode], lw=2.2, zorder=2)

    for fm in ('concrete', 'reinforcement', 'balanced'):
        mask = fm_arr == fm
        if mask.any():
            ax.scatter(xd_arr[mask], d_arr[mask],
                       marker=FM_MARKER.get(fm, 'o'), s=55,
                       facecolors=COLOR[mode], edgecolors='k', linewidths=0.8,
                       zorder=4)

ax.axhline(1.0, color='0.4', lw=1.2, ls='--')
ax.axhline(0.80, color='#1f77b4', lw=0.8, ls=':', alpha=0.5)
ax.text(0.01, 0.805, r'$\delta = 0.80$ floor', fontsize=8, color='#1f77b4', va='bottom')

ax.axhspan(1.0, 1.15, alpha=0.05, color='gray')
ax.text(0.27, 1.07, r'$A_{s,top} > A_{s,bot}$: hogging over-designed',
        fontsize=8, color='0.4', ha='center', style='italic')

ax.set_xlabel(r'$x_u/d$  (hogging section)')
ax.set_ylabel(r'$\delta_{M,u} = |M_{R,\mathrm{hog}}| \;/\; |M^\mathrm{el}_\mathrm{hog}(q_u)|$')
ax.set_xlim(-0.01, 0.53)
ax.set_ylim(0.45, 1.15)
ax.legend(handles=make_legend_handles(), fontsize=8.5, loc='lower right')
ax.set_title(
    r'Study 2: Fixed $A_{s,bot}$, varying $A_{s,top}$ — achievable $\delta_{M,u}$ vs EC2 §5.5'
    f'\n$b={b:.0f}$, $h={h:.0f}$ mm, C{f_ck:.0f}/37, B500B, $L={L/1000:.0f}$ m, '
    f'$A_{{s,bot}} = {A_s_bot_fixed:.0f}$ mm²'
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Observations — Study 2

**Left half (δ < 1, redistribution zone)**  
With $A_{s,top} \ll A_{s,bot}$, the hogging section has low capacity and fails at a low load,
but the large sagging reinforcement continues to carry load after the hogging hinge forms.
The beam redistributes heavily — $\delta_{M,u}$ can drop to 0.56 (mean) or 0.59 (design)
at small $x_u/d$.  Both values fall *below* the EC2 §5.5 floor of 0.80.  
**This does not mean EC2 is unsafe**: the §5.5 implicit route simply limits the designer to
at most 20 % redistribution, a conservative rule.  The physical capacity is larger.

**Right half (δ > 1, over-designed hogging)**  
When $A_{s,top} > A_{s,bot}$, the hogging section is stiffer than the sagging zone.
A non-homogeneous continuous beam with a stiffer interior section attracts *more* hogging
moment than the elastic homogeneous prediction $q L^2/8$ would give — hence $\delta_{M,u} > 1$.
This regime is outside the EC2 §5.5 design scenario (redistribution *reduces* hogging moment)
and is shown only for completeness.

**Design vs mean separation**  
The mean curve crosses $\delta = 1$ at $x_u/d \approx 0.21$ while the design curve crosses at
$x_u/d \approx 0.31$.  The gap reflects the ratio of mean to design moment capacities
(factor $\approx \gamma_c \gamma_s / \alpha_{cc} \approx 1.5 \times 1.15 / 0.85 \approx 2.0$).


## Combined figure — export to lecture notes

A two-panel figure combining Study 1 and Study 2 for the chapter.


In [ ]:
FIG_OUT = Path('/home/rch/Coding/icc_manuscript/lecture_notes/fig')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

for ax, results, title_suffix, show_over in [
    (axes[0], results1, r'Study 1: Symmetric $A_s$',               False),
    (axes[1], results2, rf'Study 2: Fixed $A_{{s,bot}}={A_s_bot_fixed:.0f}$ mm²', True),
]:
    ax.fill_between(xd_ec2_line, 0.45, delta_ec2_min,
                    alpha=0.12, color='#1f77b4')
    ax.plot(xd_ec2_line, delta_ec2_min, color='#1f77b4', lw=1.8)

    for mode in ('design', 'mean'):
        r = results[mode]
        xd_arr = np.array([pt['xd']      for pt in r])
        d_arr  = np.array([pt['delta']   for pt in r])
        fm_arr = np.array([pt['mk_fail'] for pt in r])

        ax.plot(xd_arr, d_arr, color=COLOR[mode], ls=STYLE[mode], lw=2.2, zorder=2)

        for fm in ('concrete', 'reinforcement', 'balanced'):
            mask = fm_arr == fm
            if mask.any():
                ax.scatter(xd_arr[mask], d_arr[mask],
                           marker=FM_MARKER.get(fm, 'o'), s=45,
                           facecolors=COLOR[mode], edgecolors='k', linewidths=0.7,
                           zorder=4)

    ax.axhline(1.0, color='0.4', lw=1.0, ls=':')
    ax.axhline(0.80, color='#1f77b4', lw=0.8, ls=':', alpha=0.5)
    ax.text(0.01, 0.805, r'$\delta = 0.80$', fontsize=8, color='#1f77b4', va='bottom')

    if show_over:
        ax.axhspan(1.0, 1.15, alpha=0.05, color='gray')
        ax.text(0.27, 1.07, r'$A_{s,top} > A_{s,bot}$: hogging over-designed',
                fontsize=8, color='0.4', ha='center', style='italic')

    ax.set_xlabel(r'$x_u/d$  (hogging section)')
    ax.set_xlim(-0.01, 0.53)
    ax.set_ylim(0.45, 1.15)
    ax.set_title(title_suffix, fontsize=11)
    ax.legend(handles=make_legend_handles(), fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel(
    r'$\delta_{M,u} = |M_{R,\mathrm{hog}}| \;/\; |M^\mathrm{el}_\mathrm{hog}(q_u)|$'
)
fig.suptitle(
    r'Simulation $\delta_{M,u}$ vs EC2 §5.5 lower bound — '
    f'$b={b:.0f}$\xa0mm, $h={h:.0f}$\xa0mm, C{f_ck:.0f}/37, B500B, $L={L/1000:.0f}$\xa0m',
    fontsize=12,
)
plt.tight_layout()

out_path = FIG_OUT / 'ch10_delta_vs_xd_nb04.png'
fig.savefig(str(out_path), dpi=150, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()
